# MedGemma 27B Oncology Text-Grounding Canary

This notebook trains a **non-deployable 512-record recovery canary** from the clean MedGemma base model. It uses TRL 0.24 native prompt/completion loss, keeps conversations separate, validates trainable labels before training, and writes to a new output directory.

Tool-call training is intentionally excluded because the current vLLM Hermes parser and Gemma chat template failed the runtime protocol gate.

In [1]:
import gc
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import cast

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.5,max_split_size_mb:256")
os.environ.setdefault("UNSLOTH_ENABLE_FLEX_ATTENTION", "0")

import unsloth
import torch
import transformers
import trl
from datasets import Dataset

PROJECT_ROOT = Path("/workspace/training/pubmed")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/home/spark/projects/training/pubmed")

BASE_LLM = "unsloth/medgemma-27b-text-it-unsloth-bnb-4bit"
MODEL_NAME_BASE = "pubmed_oncologist_recovery_text_canary_512_medgemma_sft"
INPUT_DATA_FILE = PROJECT_ROOT / "data/training-data/recovery/text_canary_512/train.jsonl"
INPUT_MANIFEST_FILE = INPUT_DATA_FILE.parent / "manifest.json"
OUTPUT_BASE_DIR = PROJECT_ROOT / "output/recovery" / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE_DIR / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / "lora_adapters"

MAX_SEQ_LENGTH = 16384
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
SAVE_STEPS = 16
WARMUP_STEPS = 2
SEED = 3407
LORA_R = 32
LORA_ALPHA = 32

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.55, 0)

print(f"unsloth={unsloth.__version__} torch={torch.__version__} transformers={transformers.__version__} trl={trl.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"Input: {INPUT_DATA_FILE}")
print(f"Output: {OUTPUT_BASE_DIR}")
assert trl.__version__ == "0.24.0", f"This notebook was verified against TRL 0.24.0, found {trl.__version__}"
assert INPUT_DATA_FILE.is_file(), f"Missing canary dataset: {INPUT_DATA_FILE}"
assert INPUT_MANIFEST_FILE.is_file(), f"Missing canary manifest: {INPUT_MANIFEST_FILE}"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth=2026.5.7 torch=2.10.0a0+b558c986e8.nv25.11 transformers=5.10.0.dev0 trl=0.24.0
GPU: NVIDIA GB10
Input: /workspace/training/pubmed/data/training-data/recovery/text_canary_512/train.jsonl
Output: /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

raw_rows = []
with INPUT_DATA_FILE.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        prompt = row.get("prompt")
        completion = row.get("completion")
        assert isinstance(prompt, list) and [message.get("role") for message in prompt] == ["system", "user"], line_number
        assert isinstance(completion, list) and len(completion) == 1, line_number
        assert completion[0].get("role") == "assistant", line_number
        assert all(isinstance(message.get("content"), str) and message["content"].strip() for message in prompt + completion), line_number
        assert "<think>" not in completion[0]["content"].lower(), line_number
        raw_rows.append(row)

manifest = json.loads(INPUT_MANIFEST_FILE.read_text(encoding="utf-8"))
input_sha256 = sha256_file(INPUT_DATA_FILE)
assert len(raw_rows) == 512, len(raw_rows)
assert manifest["rows"] == len(raw_rows)
assert manifest["output_sha256"] == input_sha256
assert manifest["tool_records_included"] is False

record_counts = {}
for row in raw_rows:
    record_counts[row["record_type"]] = record_counts.get(row["record_type"], 0) + 1
print(f"Validated {len(raw_rows)} prompt/completion records")
print(f"SHA-256: {input_sha256}")
print(f"Record types: {record_counts}")
print(json.dumps(raw_rows[0], indent=2)[:2000])

train_dataset = Dataset.from_list(raw_rows)

Validated 512 prompt/completion records
SHA-256: 0e8f5a4d0e0a2523283e380d450372ccd74bbba2d3cca306436cfcf87cdf3456
Record types: {'grounded_abstract': 480, 'missing_evidence': 16, 'general_replay': 16}
{
  "prompt": [
    {
      "role": "system",
      "content": "You are an oncology-focused medical language model for educational and research support.\n\nEvidence is conditional. Treat an abstract, tool result, or image analysis as available only when it is explicitly present in the conversation. Never invent a PMID, trial name, statistic, guideline version, measurement, retrieval result, or image finding. When evidence is missing, ask for it or state the limitation. Answer greetings and unrelated harmless questions normally without assuming a medical context. Do not provide a diagnosis or replace professional medical care."
    },
    {
      "role": "user",
      "content": "<evidence source=\"pubmed\" status=\"supplied\">\nDown-regulation of EZH2 genes targeting RUNX3 affects prolife

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print(f"Loaded {BASE_LLM}")
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={len(tokenizer)}")
print(f"pad_token={tokenizer.pad_token!r}, eos_token={tokenizer.eos_token!r}")

==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.10.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

Loaded unsloth/medgemma-27b-text-it-unsloth-bnb-4bit
Tokenizer: GemmaTokenizer, vocab=262145
pad_token='<pad>', eos_token='<end_of_turn>'


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


fatal: detected dubious ownership in repository at '/workspace/training/pubmed'
To add an exception for this directory, call:

	git config --global --add safe.directory /workspace/training/pubmed


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.10.0.dev0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


Trainable parameters: 227,033,088 / 14,610,262,784 (1.55%)


In [5]:
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

training_args = SFTConfig(
    output_dir=str(TRAIN_DIR),
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,
    dataset_num_proc=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
prepared_dataset = trainer.train_dataset
assert isinstance(prepared_dataset, Dataset), type(prepared_dataset)

print(f"Prepared rows: {len(prepared_dataset)}")
print(f"Prepared columns: {prepared_dataset.column_names}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Warmup steps: {WARMUP_STEPS}")
print("completion_only_loss=True, packing=False")

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/multiprocess/popen_fork.py:66: DeprecationWarning: This process (pid=234) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Prepared rows: 512
Prepared columns: ['input_ids', 'completion_mask']
Effective batch: 8
Warmup steps: 2
completion_only_loss=True, packing=False


In [6]:
def contains_subsequence(sequence: list[int], subsequence: list[int]) -> bool:
    if not subsequence or len(subsequence) > len(sequence):
        return False
    width = len(subsequence)
    return any(sequence[index:index + width] == subsequence for index in range(len(sequence) - width + 1))

representative_indices = {}
for index, row in enumerate(raw_rows):
    representative_indices.setdefault(row["record_type"], index)

collate_batch = trainer.data_collator
assert callable(collate_batch), "Trainer did not configure a data collator"
mask_report = []
for record_type, index in sorted(representative_indices.items()):
    processed = prepared_dataset[index]
    batch = collate_batch([processed])
    labels = batch["labels"][0]
    input_ids = batch["input_ids"][0]
    trainable_mask = labels.ne(-100)
    assert trainable_mask.any(), f"No trainable labels for {record_type}"

    prompt_ids = tokenizer.apply_chat_template(
        raw_rows[index]["prompt"],
        tokenize=True,
        add_generation_prompt=True,
    )
    assert labels[:len(prompt_ids)].eq(-100).all(), f"Prompt tokens are trainable for {record_type}"

    trainable_ids = labels[trainable_mask].tolist()
    completion_text = raw_rows[index]["completion"][0]["content"]
    completion_ids = tokenizer.encode(completion_text, add_special_tokens=False)
    assert contains_subsequence(trainable_ids, completion_ids), f"Completion is not fully trainable for {record_type}"

    decoded_labels = tokenizer.decode(trainable_ids, skip_special_tokens=False)
    user_text = raw_rows[index]["prompt"][-1]["content"]
    user_prefix_ids = tokenizer.encode(user_text[:120], add_special_tokens=False)
    assert not contains_subsequence(trainable_ids, user_prefix_ids), f"User text leaked into labels for {record_type}"

    mask_report.append(
        {
            "record_type": record_type,
            "sequence_tokens": int(input_ids.ne(tokenizer.pad_token_id).sum()),
            "trainable_tokens": int(trainable_mask.sum()),
            "trainable_percent": round(100 * trainable_mask.float().mean().item(), 2),
            "decoded_label_preview": decoded_labels[:300],
        }
    )

print(json.dumps(mask_report, indent=2))
print("LABEL MASK GATE PASSED: prompts are masked and only completions are trainable.")

[
  {
    "record_type": "general_replay",
    "sequence_tokens": 127,
    "trainable_tokens": 10,
    "trainable_percent": 7.87,
    "decoded_label_preview": "Please provide the text you want summarized.<end_of_turn>\n"
  },
  {
    "record_type": "grounded_abstract",
    "sequence_tokens": 366,
    "trainable_tokens": 36,
    "trainable_percent": 9.84,
    "decoded_label_preview": "Down-regulation of EZH2 genes targeting RUNX3 affects proliferation, invasion, and metastasis of human colon cancer cells by Wnt/\u03b2-catenin signaling pathway.<end_of_turn>\n"
  },
  {
    "record_type": "missing_evidence",
    "sequence_tokens": 152,
    "trainable_tokens": 30,
    "trainable_percent": 19.74,
    "decoded_label_preview": "I don't have the study text or a tool result, so I cannot extract its design or response rates. Please supply the evidence.<end_of_turn>\n"
  }
]
LABEL MASK GATE PASSED: prompts are masked and only completions are trainable.


In [7]:
from transformers.trainer_utils import get_last_checkpoint

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
last_checkpoint = get_last_checkpoint(str(TRAIN_DIR))
if last_checkpoint:
    print(f"Resuming from {last_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting the canary from the clean base model")
    train_result = trainer.train()

print(f"Training complete: global_step={train_result.global_step}")
print(train_result.metrics)

Starting the canary from the clean base model


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 512 | Num Epochs = 1 | Total steps = 64
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 227,033,088 of 27,236,035,328 (0.83% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.950925
2,0.997993
3,0.205228
4,0.043758
5,0.135238
6,0.088151
7,0.146017
8,0.069291
9,0.048858
10,0.083401


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/train/checkpoint-16/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/train/checkpoint-16.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/train/checkpoint-32/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/train/checkpoint-32.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/train/checkpoint-48/tokenizer

Training complete: global_step=64
{'train_runtime': 1192.3924, 'train_samples_per_second': 0.429, 'train_steps_per_second': 0.054, 'total_flos': 4.165895966177894e+16, 'train_loss': 0.12871067198284436, 'epoch': 1.0}


In [8]:
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

last_checkpoint = get_last_checkpoint(str(TRAIN_DIR))
complete = {
    "status": "trained_pending_behavioral_evaluation",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "input_data_file": str(INPUT_DATA_FILE),
    "input_sha256": input_sha256,
    "input_manifest_file": str(INPUT_MANIFEST_FILE),
    "output_dir": str(OUTPUT_BASE_DIR),
    "lora_output_dir": str(LORA_OUTPUT_DIR),
    "last_checkpoint": last_checkpoint,
    "global_step": train_result.global_step,
    "num_train_epochs": NUM_EPOCHS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "completion_only_loss": True,
    "packing": False,
    "tool_records_included": False,
    "package_versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
    },
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
(LORA_OUTPUT_DIR / "complete.json").write_text(json.dumps(complete, indent=2) + "\n", encoding="utf-8")
print(json.dumps(complete, indent=2))

[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/lora_adapters/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/lora_adapters.


{
  "status": "trained_pending_behavioral_evaluation",
  "model_name": "pubmed_oncologist_recovery_text_canary_512_medgemma_sft",
  "base_model": "unsloth/medgemma-27b-text-it-unsloth-bnb-4bit",
  "input_data_file": "/workspace/training/pubmed/data/training-data/recovery/text_canary_512/train.jsonl",
  "input_sha256": "0e8f5a4d0e0a2523283e380d450372ccd74bbba2d3cca306436cfcf87cdf3456",
  "input_manifest_file": "/workspace/training/pubmed/data/training-data/recovery/text_canary_512/manifest.json",
  "output_dir": "/workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft",
  "lora_output_dir": "/workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/lora_adapters",
  "last_checkpoint": "/workspace/training/pubmed/output/recovery/pubmed_oncologist_recovery_text_canary_512_medgemma_sft/train/checkpoint-64",
  "global_step": 64,
  "num_train_epochs": 1,
  "max_seq_length": 16384,
  "completion_only_loss": tr

In [ ]:
FastLanguageModel.for_inference(model)

SMOKE_PROMPTS = [
    "Hi",
    "Summarize the PubMed abstract I provided.",
    "What can you help me with?",
]
for user_prompt in SMOKE_PROMPTS:
    messages = [
        {"role": "system", "content": raw_rows[0]["prompt"][0]["content"]},
        {"role": "user", "content": user_prompt},
    ]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(
        rendered,
        return_tensors="pt",
        add_special_tokens=False,
    )
    encoded = {name: tensor.to(model.device) for name, tensor in encoded.items()}
    with torch.inference_mode():
        output = cast(
            torch.Tensor,
            model.generate(
                **encoded,
                max_new_tokens=160,
                do_sample=False,
                use_cache=True,
            ),
        )
    response = tokenizer.decode(
        output[0, encoded["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    print(f"USER: {user_prompt}\nASSISTANT: {response}\n")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER: Hi
ASSISTANT: Hello! How can I help you with oncology-related information or research today?



[transformers] Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER: Summarize the PubMed abstract I provided.
ASSISTANT: No abstract is present. Please provide it before asking for a summary.

USER: What can you help me with?
ASSISTANT: I can help you understand oncology-related text, identify key concepts, summarize findings, and locate relevant information within provided documents. I can also assist with research tasks like identifying potential study targets or generating hypotheses based on the available data.



In [ ]:
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

reloaded_model, reloaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(reloaded_model)

reload_messages = [
    {"role": "system", "content": raw_rows[0]["prompt"][0]["content"]},
    {"role": "user", "content": "Summarize the abstract I supplied."},
]
reload_rendered = reloaded_tokenizer.apply_chat_template(
    reload_messages,
    tokenize=False,
    add_generation_prompt=True,
)
reload_encoded = reloaded_tokenizer(
    reload_rendered,
    return_tensors="pt",
    add_special_tokens=False,
)
reload_encoded = {
    name: tensor.to(reloaded_model.device)
    for name, tensor in reload_encoded.items()
}
with torch.inference_mode():
    reload_output = cast(
        torch.Tensor,
        reloaded_model.generate(
            **reload_encoded,
            max_new_tokens=120,
            do_sample=False,
            use_cache=True,
        ),
    )
reload_response = reloaded_tokenizer.decode(
    reload_output[0, reload_encoded["input_ids"].shape[-1]:],
    skip_special_tokens=True,
).strip()
assert reload_response, "Cold-reloaded adapter produced an empty response"
print(f"Cold reload passed.\nASSISTANT: {reload_response}")

==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.10.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=234) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Cold reload passed.
ASSISTANT: No abstract is present. Please provide it.
